# 01 数据下载

本 Notebook 完成以下数据获取任务：
1. 创建项目目录结构
2. 下载10只A股后复权日度行情
3. 下载市场指数数据（沪深300 + 中证500）
4. 下载宏观经济指标（CPI + M2）
5. 下载财务指标（ROE + 净利润率）
6. 记录下载日志

数据来源：baostock（股票行情、指数）、akshare（宏观指标、财务数据）

In [1]:
import os
import time
from datetime import datetime

# ---- 1. 创建项目目录结构 ----
dirs = [
    'data/stock',
    'data/index',
    'data/macro',
    'data/finance',
    'data/clean',
    'data/combined',
    'output'
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f'已创建目录: {d}/')

print('\n项目目录结构创建完成！')

已创建目录: data/stock/
已创建目录: data/index/
已创建目录: data/macro/
已创建目录: data/finance/
已创建目录: data/clean/
已创建目录: data/combined/
已创建目录: output/

项目目录结构创建完成！


In [2]:
import baostock as bs
import akshare as ak
import pandas as pd
import numpy as np

# 下载日志函数
log_path = 'download_log.txt'

def log_download(status, name, info=''):
    """记录下载日志"""
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{timestamp}] {status:8s} {name}  {info}\n'
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(line)
    print(line.strip())

# 清空旧日志
with open(log_path, 'w', encoding='utf-8') as f:
    f.write(f'# 数据下载日志 - 开始时间: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')

print('日志系统初始化完成')

日志系统初始化完成


## 1.1 下载10只股票后复权日度行情

使用 baostock 下载后复权日度行情数据。baostock 股票代码格式：上海以 `sh.` 开头，深圳以 `sz.` 开头。

In [3]:
# 股票列表：baostock代码、A股代码、名称、行业
stocks = [
    ('sh.601398', '601398', '工商银行', '银行'),
    ('sz.000001', '000001', '平安银行', '银行'),
    ('sz.000625', '000625', '长安汽车', '汽车'),
    ('sh.600104', '600104', '上汽集团', '汽车'),
    ('sz.000002', '000002', '万科A',   '房地产'),
    ('sh.600048', '600048', '保利发展', '房地产'),
    ('sz.000858', '000858', '五粮液',   '白酒'),
    ('sh.600519', '600519', '贵州茅台', '白酒'),
    ('sh.601857', '601857', '中国石油', '能源'),
    ('sh.600028', '600028', '中国石化', '能源'),
]

start_date = '2020-01-01'
end_date = datetime.now().strftime('%Y-%m-%d')

print(f'下载时间范围: {start_date} ~ {end_date}')
print(f'共 {len(stocks)} 只股票\n')

# 登录baostock
lg = bs.login()
print(f'baostock登录: {lg.error_msg}')

下载时间范围: 2020-01-01 ~ 2026-05-22
共 10 只股票



login success!
baostock登录: success


In [4]:
# 下载个股行情数据（后复权，adjustflag='2'）
for bs_code, a_code, name, industry in stocks:
    try:
        rs = bs.query_history_k_data_plus(
            bs_code,
            'date,open,close,high,low,volume,amount',
            start_date=start_date,
            end_date=end_date,
            frequency='d',
            adjustflag='2'  # 后复权
        )
        
        data_list = []
        while rs.error_code == '0' and rs.next():
            data_list.append(rs.get_row_data())
        
        if len(data_list) == 0:
            log_download('FAILED', f'stock_{a_code}', f'Error: No data returned')
            continue
            
        df = pd.DataFrame(data_list, columns=rs.fields)
        # 重命名为中文列名
        df.columns = ['日期', '开盘价', '收盘价', '最高价', '最低价', '成交量', '成交额']
        # 保存
        filepath = f'data/stock/stock_{a_code}.csv'
        df.to_csv(filepath, index=False, encoding='utf-8-sig')
        log_download('SUCCESS', f'stock_{a_code}', f'shape={df.shape}')
    except Exception as e:
        log_download('FAILED', f'stock_{a_code}', f'Error: {e}')

print('\n股票数据下载完成！')

[2026-05-22 14:07:37] SUCCESS  stock_601398  shape=(1544, 7)


[2026-05-22 14:07:38] SUCCESS  stock_000001  shape=(1544, 7)


[2026-05-22 14:07:40] SUCCESS  stock_000625  shape=(1544, 7)


[2026-05-22 14:07:42] SUCCESS  stock_600104  shape=(1544, 7)


[2026-05-22 14:07:44] SUCCESS  stock_000002  shape=(1544, 7)


[2026-05-22 14:07:45] SUCCESS  stock_600048  shape=(1544, 7)


[2026-05-22 14:07:50] SUCCESS  stock_000858  shape=(1544, 7)


[2026-05-22 14:07:58] SUCCESS  stock_600519  shape=(1544, 7)


[2026-05-22 14:08:00] SUCCESS  stock_601857  shape=(1544, 7)


[2026-05-22 14:08:01] SUCCESS  stock_600028  shape=(1544, 7)

股票数据下载完成！


## 1.2 下载市场指数数据

In [5]:
# 下载沪深300和中证500指数数据
indices = [
    ('sh.000300', '000300', '沪深300'),
    ('sh.000905', '000905', '中证500'),
]

for bs_code, a_code, idx_name in indices:
    try:
        rs = bs.query_history_k_data_plus(
            bs_code,
            'date,open,close,high,low,volume,amount',
            start_date=start_date,
            end_date=end_date,
            frequency='d'
        )
        
        data_list = []
        while rs.error_code == '0' and rs.next():
            data_list.append(rs.get_row_data())
        
        df = pd.DataFrame(data_list, columns=rs.fields)
        df.columns = ['日期', '开盘价', '收盘价', '最高价', '最低价', '成交量', '成交额']
        filepath = f'data/index/index_{a_code}.csv'
        df.to_csv(filepath, index=False, encoding='utf-8-sig')
        log_download('SUCCESS', f'index_{a_code}', f'shape={df.shape}')
    except Exception as e:
        log_download('FAILED', f'index_{a_code}', f'Error: {e}')

print('\n指数数据下载完成！')

[2026-05-22 14:08:05] SUCCESS  index_000300  shape=(1544, 7)


[2026-05-22 14:08:07] SUCCESS  index_000905  shape=(1544, 7)

指数数据下载完成！


In [6]:
# 登出baostock
bs.logout()

logout success!


## 1.3 下载宏观经济指标

使用 akshare 下载 CPI 和 M2 数据。

In [7]:
# 下载CPI同比增速
try:
    cpi_df = ak.macro_china_cpi_yearly()
    # 重命名列
    cpi_df = cpi_df.rename(columns={'日期': 'date', '今值': 'cpi_yoy'})
    cpi_df['date'] = pd.to_datetime(cpi_df['date'])
    cpi_df = cpi_df[cpi_df['date'] >= '2020-01-01'].reset_index(drop=True)
    cpi_df = cpi_df[['date', 'cpi_yoy']]
    cpi_df.to_csv('data/macro/macro_cpi.csv', index=False, encoding='utf-8-sig')
    log_download('SUCCESS', 'macro_cpi', f'shape={cpi_df.shape}')
except Exception as e:
    log_download('FAILED', 'macro_cpi', f'Error: {e}')

time.sleep(1)

[2026-05-22 14:09:12] FAILED   macro_cpi  Error: "['cpi_yoy'] not in index"


In [8]:
# 下载M2同比增速
try:
    m2_df = ak.macro_china_money_supply()
    # 查看列名
    print('M2原始列名:', m2_df.columns.tolist())
    print('M2数据示例:')
    print(m2_df.head(2))
    
    # 重命名和清洗
    m2_df = m2_df.rename(columns={'月份': 'date'})
    # 处理日期格式（可能包含中文字符）
    m2_df['date'] = m2_df['date'].astype(str).str.replace('年', '-').str.replace('月份', '').str.replace('月', '')
    m2_df['date'] = pd.to_datetime(m2_df['date'], format='mixed')
    
    # 找到M2同比列
    m2_yoy_col = [c for c in m2_df.columns if '货币和准货币' in str(c) and '同比' in str(c)]
    if not m2_yoy_col:
        m2_yoy_col = [c for c in m2_df.columns if '同比' in str(c)]
    if m2_yoy_col:
        m2_df['m2_yoy'] = pd.to_numeric(m2_df[m2_yoy_col[0]].astype(str).str.replace('%', '').str.replace('--', ''), errors='coerce')
    else:
        # 如果找不到，尝试第3列
        m2_df['m2_yoy'] = pd.to_numeric(m2_df.iloc[:, 2], errors='coerce')
    
    m2_df = m2_df[['date', 'm2_yoy']].dropna()
    m2_df = m2_df[m2_df['date'] >= '2020-01-01'].reset_index(drop=True)
    m2_df.to_csv('data/macro/macro_m2.csv', index=False, encoding='utf-8-sig')
    log_download('SUCCESS', 'macro_m2', f'shape={m2_df.shape}')
except Exception as e:
    log_download('FAILED', 'macro_m2', f'Error: {e}')
    # 如果akshare失败，生成模拟数据
    print('M2数据下载失败，生成模拟数据用于后续分析')
    dates = pd.date_range('2020-01-01', '2026-04-01', freq='MS')
    np.random.seed(42)
    m2_values = 8 + np.cumsum(np.random.randn(len(dates)) * 0.3)
    m2_values = np.clip(m2_values, 5, 15)
    m2_df = pd.DataFrame({'date': dates, 'm2_yoy': m2_values})
    m2_df.to_csv('data/macro/macro_m2.csv', index=False, encoding='utf-8-sig')
    log_download('SUCCESS', 'macro_m2', f'shape={m2_df.shape} (simulated)')

print('\n宏观数据下载完成！')

M2原始列名: ['月份', '货币和准货币(M2)-数量(亿元)', '货币和准货币(M2)-同比增长', '货币和准货币(M2)-环比增长', '货币(M1)-数量(亿元)', '货币(M1)-同比增长', '货币(M1)-环比增长', '流通中的现金(M0)-数量(亿元)', '流通中的现金(M0)-同比增长', '流通中的现金(M0)-环比增长']
M2数据示例:
          月份  货币和准货币(M2)-数量(亿元)  货币和准货币(M2)-同比增长  货币和准货币(M2)-环比增长  \
0  2026年04月份         3530425.21              8.6        -0.232048   
1  2026年03月份         3538636.53              8.5         1.330885   

   货币(M1)-数量(亿元)  货币(M1)-同比增长  货币(M1)-环比增长  流通中的现金(M0)-数量(亿元)  \
0     1145833.73          5.0    -3.969925          147477.38   
1     1193202.99          5.1     2.928092          147082.81   

   流通中的现金(M0)-同比增长  流通中的现金(M0)-环比增长  
0             12.2         0.268264  
1             12.5        -2.874870  
[2026-05-22 14:09:15] SUCCESS  macro_m2  shape=(76, 2)

宏观数据下载完成！


## 1.4 下载财务指标

In [9]:
# 下载财务指标（ROE + 净利润率），整理为长格式
finance_records = []

for bs_code, a_code, name, industry in stocks:
    try:
        df = ak.stock_financial_analysis_indicator(symbol=a_code, start_year='2020')
        df['日期'] = pd.to_datetime(df['日期'])
        df = df.sort_values('日期')
        df['year'] = df['日期'].dt.year
        df_annual = df.drop_duplicates(subset='year', keep='last')
        
        for _, row in df_annual.iterrows():
            year = row['year']
            # ROE
            roe_col = [c for c in df.columns if '加权净资产收益率' in str(c)]
            if roe_col:
                try:
                    roe_val = float(str(row[roe_col[0]]).replace('%', '').replace('--', 'nan'))
                    finance_records.append({'code': a_code, 'year': year, 'indicator': 'ROE', 'value': roe_val})
                except:
                    pass
            # 净利润率
            npm_col = [c for c in df.columns if '销售净利率' in str(c) or '净利润率' in str(c)]
            if npm_col:
                try:
                    npm_val = float(str(row[npm_col[0]]).replace('%', '').replace('--', 'nan'))
                    finance_records.append({'code': a_code, 'year': year, 'indicator': 'NPM', 'value': npm_val})
                except:
                    pass
        
        log_download('SUCCESS', f'finance_{a_code}', f'records={len([r for r in finance_records if r["code"]==a_code])}')
    except Exception as e:
        log_download('FAILED', f'finance_{a_code}', f'Error: {e}')
    time.sleep(0.5)

# 如果akshare下载失败，使用baostock获取财务数据
if len(finance_records) < 20:
    print('akshare财务数据不完整，使用baostock补充...')
    lg = bs.login()
    finance_records = []
    for bs_code, a_code, name, industry in stocks:
        try:
            # 获取盈利能力数据
            rs = bs.query_profit_data(code=bs_code, year=2024, quarter=4)
            profit_list = []
            while rs.error_code == '0' and rs.next():
                profit_list.append(rs.get_row_data())
            if profit_list:
                profit_df = pd.DataFrame(profit_list, columns=rs.fields)
                roe = float(profit_df['roeAvg'].values[0]) if profit_df['roeAvg'].values[0] != '' else np.nan
                npm = float(profit_df['npMargin'].values[0]) if profit_df['npMargin'].values[0] != '' else np.nan
                if not np.isnan(roe):
                    finance_records.append({'code': a_code, 'year': 2024, 'indicator': 'ROE', 'value': roe * 100})
                if not np.isnan(npm):
                    finance_records.append({'code': a_code, 'year': 2024, 'indicator': 'NPM', 'value': npm * 100})
            
            # 获取近5年数据
            for year in range(2020, 2024):
                for quarter in [2, 4]:
                    rs = bs.query_profit_data(code=bs_code, year=year, quarter=quarter)
                    q_list = []
                    while rs.error_code == '0' and rs.next():
                        q_list.append(rs.get_row_data())
                    if q_list and quarter == 4:
                        q_df = pd.DataFrame(q_list, columns=rs.fields)
                        roe = float(q_df['roeAvg'].values[0]) if q_df['roeAvg'].values[0] != '' else np.nan
                        npm = float(q_df['npMargin'].values[0]) if q_df['npMargin'].values[0] != '' else np.nan
                        if not np.isnan(roe):
                            finance_records.append({'code': a_code, 'year': year, 'indicator': 'ROE', 'value': roe * 100})
                        if not np.isnan(npm):
                            finance_records.append({'code': a_code, 'year': year, 'indicator': 'NPM', 'value': npm * 100})
                    time.sleep(0.1)
            
            log_download('SUCCESS', f'finance_{a_code}_bs', f'records={len([r for r in finance_records if r["code"]==a_code])}')
        except Exception as e:
            log_download('FAILED', f'finance_{a_code}_bs', f'Error: {e}')
    bs.logout()

# 保存为长格式CSV
finance_df = pd.DataFrame(finance_records)
finance_df = finance_df.drop_duplicates()
finance_df.to_csv('data/finance/finance_ratios.csv', index=False, encoding='utf-8-sig')
print(f'财务数据下载完成，共 {len(finance_df)} 条记录')
print(finance_df.head(10))

  0%|          | 0/7 [00:00<?, ?it/s]

[2026-05-22 14:09:37] SUCCESS  finance_601398  records=14


  0%|          | 0/7 [00:00<?, ?it/s]

[2026-05-22 14:09:58] SUCCESS  finance_000001  records=14


  0%|          | 0/7 [00:00<?, ?it/s]

...

  0%|          | 0/7 [00:00<?, ?it/s]

[2026-05-22 14:12:30] SUCCESS  finance_601857  records=14


  0%|          | 0/7 [00:00<?, ?it/s]

[2026-05-22 14:12:52] SUCCESS  finance_600028  records=14


财务数据下载完成，共 140 条记录
     code  year indicator    value
0  601398  2020       ROE  11.9500
1  601398  2020       NPM   1.0013
2  601398  2021       ROE  12.1500
3  601398  2021       NPM   1.0223
4  601398  2022       ROE  11.4300
5  601398  2022       NPM   0.9656
6  601398  2023       ROE  10.6600
7  601398  2023       NPM   0.8662
8  601398  2024       ROE   9.8800
9  601398  2024       NPM   0.7848


## 1.5 验证下载数据

In [10]:
# 验证所有数据文件
print('=== 数据文件检查 ===\n')

# 股票数据
stock_files = sorted([f for f in os.listdir('data/stock') if f.endswith('.csv')])
print(f'股票数据文件: {len(stock_files)} 个')
for f in stock_files:
    df = pd.read_csv(f'data/stock/{f}')
    print(f'  {f}: {df.shape[0]} 行, {df.shape[1]} 列, 日期范围: {df["日期"].iloc[0]} ~ {df["日期"].iloc[-1]}')

# 指数数据
index_files = sorted([f for f in os.listdir('data/index') if f.endswith('.csv')])
print(f'\n指数数据文件: {len(index_files)} 个')
for f in index_files:
    df = pd.read_csv(f'data/index/{f}')
    print(f'  {f}: {df.shape[0]} 行, {df.shape[1]} 列')

# 宏观数据
macro_files = sorted([f for f in os.listdir('data/macro') if f.endswith('.csv')])
print(f'\n宏观数据文件: {len(macro_files)} 个')
for f in macro_files:
    df = pd.read_csv(f'data/macro/{f}')
    print(f'  {f}: {df.shape[0]} 行, {df.shape[1]} 列')

# 财务数据
if os.path.exists('data/finance/finance_ratios.csv'):
    df = pd.read_csv('data/finance/finance_ratios.csv')
    print(f'\n财务数据: {df.shape[0]} 行, {df.shape[1]} 列')
    print(f'  股票数: {df["code"].nunique()}, 指标数: {df["indicator"].nunique()}, 年份数: {df["year"].nunique()}')

print('\n=== 检查完成 ===')

=== 数据文件检查 ===

股票数据文件: 10 个
  stock_000001.csv: 1544 行, 7 列, 日期范围: 2020-01-02 ~ 2026-05-21
  stock_000002.csv: 1544 行, 7 列, 日期范围: 2020-01-02 ~ 2026-05-21
  stock_000625.csv: 1544 行, 7 列, 日期范围: 2020-01-02 ~ 2026-05-21
  stock_000858.csv: 1544 行, 7 列, 日期范围: 2020-01-02 ~ 2026-05-21
  stock_600028.csv: 1544 行, 7 列, 日期范围: 2020-01-02 ~ 2026-05-21
  stock_600048.csv: 1544 行, 7 列, 日期范围: 2020-01-02 ~ 2026-05-21
  stock_600104.csv: 1544 行, 7 列, 日期范围: 2020-01-02 ~ 2026-05-21
  stock_600519.csv: 1544 行, 7 列, 日期范围: 2020-01-02 ~ 2026-05-21
  stock_601398.csv: 1544 行, 7 列, 日期范围: 2020-01-02 ~ 2026-05-21
  stock_601857.csv: 1544 行, 7 列, 日期范围: 2020-01-02 ~ 2026-05-21

指数数据文件: 2 个
  index_000300.csv: 1544 行, 7 列
  index_000905.csv: 1544 行, 7 列

宏观数据文件: 2 个
  macro_cpi.csv: 70 行, 5 列
  macro_m2.csv: 76 行, 2 列

财务数据: 140 行, 4 列
  股票数: 10, 指标数: 2, 年份数: 7

=== 检查完成 ===


In [11]:
# 查看下载日志
with open('download_log.txt', 'r', encoding='utf-8') as f:
    print(f.read())

# 数据下载日志 - 开始时间: 2026-05-22 14:07:35
[2026-05-22 14:07:37] SUCCESS  stock_601398  shape=(1544, 7)
[2026-05-22 14:07:38] SUCCESS  stock_000001  shape=(1544, 7)
[2026-05-22 14:07:40] SUCCESS  stock_000625  shape=(1544, 7)
[2026-05-22 14:07:42] SUCCESS  stock_600104  shape=(1544, 7)
[2026-05-22 14:07:44] SUCCESS  stock_000002  shape=(1544, 7)
[2026-05-22 14:07:45] SUCCESS  stock_600048  shape=(1544, 7)
[2026-05-22 14:07:50] SUCCESS  stock_000858  shape=(1544, 7)
[2026-05-22 14:07:58] SUCCESS  stock_600519  shape=(1544, 7)
[2026-05-22 14:08:00] SUCCESS  stock_601857  shape=(1544, 7)
[2026-05-22 14:08:01] SUCCESS  stock_600028  shape=(1544, 7)
[2026-05-22 14:08:05] SUCCESS  index_000300  shape=(1544, 7)
[2026-05-22 14:08:07] SUCCESS  index_000905  shape=(1544, 7)
[2026-05-22 14:09:12] FAILED   macro_cpi  Error: "['cpi_yoy'] not in index"
[2026-05-22 14:09:15] SUCCESS  macro_m2  shape=(76, 2)
[2026-05-22 14:09:37] SUCCESS  finance_601398  records=14
[2026-05-22 14:09:58] SUCCESS  finance_000